# パフォーマンスメトリクス分析

Prometheus で収集した v18 パフォーマンスメトリクスの分析

- リクエスト数・エラー率
- レイテンシー分布（P50, P95, P99）
- ロール別パフォーマンス
- ルート別統計
- タイムシリーズ分析

In [ ]:
import os
import urllib.request
import json
import pandas as pd
from datetime import datetime, timedelta

# Prometheus エンドポイント
PROMETHEUS_URL = os.environ.get('PROMETHEUS_URL', 'http://192.168.11.231:9090')

print(f"Prometheus: {PROMETHEUS_URL}")

def query_prometheus(query, time_range='1h'):
    """Prometheus クエリを実行"""
    try:
        url = f"{PROMETHEUS_URL}/api/v1/query?query={query}&time={datetime.now().timestamp()}"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=10) as resp:
            return json.loads(resp.read().decode('utf-8'))
    except Exception as e:
        print(f"クエリエラー: {e}")
        return {}

print("✅ Prometheus接続準備完了")

In [ ]:
# v18 メトリクスのサマリー
print("=== v18 パフォーマンスメトリクス ===")

# リクエスト総数
result = query_prometheus('bushidan_requests_total')
if result.get('data', {}).get('result'):
    total_requests = sum(float(r['value'][1]) for r in result['data']['result'])
    print(f"\n総リクエスト数: {total_requests:.0f}")
    
    # ソース別
    by_source = {}
    for r in result['data']['result']:
        source = r['metric'].get('source', 'unknown')
        count = float(r['value'][1])
        by_source[source] = by_source.get(source, 0) + count
    
    print("  ソース別:")
    for source, count in sorted(by_source.items(), key=lambda x: x[1], reverse=True):
        print(f"    {source}: {count:.0f}")

# エラー数
result = query_prometheus('bushidan_errors_total')
if result.get('data', {}).get('result'):
    total_errors = sum(float(r['value'][1]) for r in result['data']['result'])
    print(f"\n総エラー数: {total_errors:.0f}")
    if total_requests > 0:
        print(f"エラー率: {total_errors/total_requests*100:.2f}%")

In [ ]:
# レイテンシー分析
print("=== レイテンシー分析 ===")

result = query_prometheus('bushidan_request_duration_seconds')
if result.get('data', {}).get('result'):
    latencies = []
    for r in result['data']['result']:
        for val_pair in r.get('values', []):
            if len(val_pair) > 1:
                latencies.append(float(val_pair[1]))
    
    if latencies:
        latencies.sort()
        print(f"\nサンプル数: {len(latencies)}")
        print(f"最小: {min(latencies):.3f}s")
        print(f"P50: {latencies[int(len(latencies)*0.5)]:.3f}s")
        print(f"P95: {latencies[int(len(latencies)*0.95)]:.3f}s")
        print(f"P99: {latencies[int(len(latencies)*0.99)]:.3f}s")
        print(f"最大: {max(latencies):.3f}s")
        print(f"平均: {sum(latencies)/len(latencies):.3f}s")

In [ ]:
# アクティブWebSocket接続数
print("=== WebSocket接続状態 ===")

result = query_prometheus('bushidan_active_websockets')
if result.get('data', {}).get('result'):
    for r in result['data']['result']:
        count = float(r['value'][1])
        print(f"アクティブ接続: {count:.0f}")

In [ ]:
# ルート別統計
print("\n=== ルート別統計 ===")

result = query_prometheus('bushidan_requests_total')
if result.get('data', {}).get('result'):
    by_route = {}
    for r in result['data']['result']:
        route = r['metric'].get('route', 'unknown')
        count = float(r['value'][1])
        by_route[route] = by_route.get(route, 0) + count
    
    print("\nルート別リクエスト数（TOP10）:")
    for route, count in sorted(by_route.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {route}: {count:.0f}")

In [ ]:
# グラフ表示
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ソース別リクエスト分布
if 'by_source' in locals():
    pd.Series(by_source).plot(kind='pie', ax=axes[0, 0], autopct='%1.1f%%')
    axes[0, 0].set_title('リクエスト ソース別分布')
    axes[0, 0].set_ylabel('')

# ルート別リクエスト分布
if 'by_route' in locals():
    top_routes = sorted(by_route.items(), key=lambda x: x[1], reverse=True)[:8]
    pd.Series(dict(top_routes)).plot(kind='barh', ax=axes[0, 1], color='steelblue')
    axes[0, 1].set_title('ルート別リクエスト数（TOP8）')
    axes[0, 1].set_xlabel('リクエスト数')

# レイテンシー分布
if 'latencies' in locals() and latencies:
    axes[1, 0].hist(latencies, bins=30, edgecolor='black', color='coral')
    axes[1, 0].axvline(np.percentile(latencies, 50), color='red', linestyle='--', label='P50')
    axes[1, 0].axvline(np.percentile(latencies, 95), color='orange', linestyle='--', label='P95')
    axes[1, 0].set_title('レイテンシー分布')
    axes[1, 0].set_xlabel('レイテンシー (s)')
    axes[1, 0].set_ylabel('頻度')
    axes[1, 0].legend()

# エラー率
if 'total_errors' in locals() and 'total_requests' in locals() and total_requests > 0:
    error_rate = total_errors / total_requests * 100
    success_rate = 100 - error_rate
    axes[1, 1].pie([success_rate, error_rate], labels=['成功', 'エラー'], 
                    autopct='%1.2f%%', colors=['#90EE90', '#FFB6C6'])
    axes[1, 1].set_title(f'成功率 ({error_rate:.2f}% エラー)')

plt.tight_layout()
plt.show()

print("✅ メトリクスグラフ表示完了")

In [ ]:
# 推奨事項
print("\n=== パフォーマンス分析 & 推奨 ===")

if 'latencies' in locals() and latencies:
    p95 = np.percentile(latencies, 95)
    if p95 > 5:  # 5秒以上
        print("⚠️ P95レイテンシーが高い (> 5s) — LLM呼び出しのタイムアウト設定を確認")
    elif p95 > 10:  # 10秒以上
        print("⚠️ P95レイテンシーが非常に高い (> 10s) — パイプラインの最適化が必要")
    else:
        print(f"✅ P95レイテンシー良好: {p95:.2f}s")

if 'total_errors' in locals() and total_requests > 0:
    error_rate = total_errors / total_requests * 100
    if error_rate > 5:
        print(f"⚠️ エラー率が高い: {error_rate:.2f}%")
    else:
        print(f"✅ エラー率低い: {error_rate:.2f}%")